In [1]:
import os, json, random, time
import numpy as np
import cv2

# Works on Windows (TF) and Pi (tflite-runtime)
try:
    import tflite_runtime.interpreter as tflite
    USING = "tflite_runtime"
except ImportError:
    import tensorflow.lite as tflite
    USING = "tensorflow.lite"

# ============================================================
#  TUNE THRESHOLD USING COMPACT DB (OPTION B)
#  - Uses UNKNOWN_DIR to compute best similarity vs compact DB
#  - Sets threshold as quantile (e.g., 0.99 => reject ~99% unknown)
# ============================================================

# ================== CONFIG ==================
MODEL_PATH = r"models\facenet.tflite"

# Compact DB (centroids / prototypes)
DB_COMPACT_PATH = r"face_db_compact.npz"

# Unknown folder containing random people images (flat images)
UNKNOWN_DIR = r"unknown"   # <-- set your relocated unknown path

# Output threshold file
THRESH_OUT = r"threshold_compact.json"

# Quantile to reject unknown
UNKNOWN_REJECT_QUANTILE = 0.99

# Face detection
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"

# Detection speed-up
DETECT_DOWNSCALE_MAX_W = 640

# Use ALL unknown images (set None for all)
MAX_UNKNOWN_IMGS = None

# Progress printing
PRINT_EVERY = 50

RANDOM_SEED = 42
# ============================================


# ----------------------------
# Helpers
# ----------------------------
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

face_cascade = cv2.CascadeClassifier(CASCADE_PATH)

def list_images(folder):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [f for f in os.listdir(folder) if f.lower().endswith(exts)]

def l2_normalize(v, eps=1e-10):
    return v / (np.linalg.norm(v) + eps)

def print_progress(prefix, i, total, extra=""):
    if total <= 0:
        return
    pct = (i / total) * 100.0
    msg = f"\r{prefix}: {i}/{total} ({pct:5.1f}%) {extra}"
    print(msg, end="", flush=True)
    if i == total:
        print()

def detect_biggest_face(bgr):
    h, w = bgr.shape[:2]
    scale = 1.0
    if w > DETECT_DOWNSCALE_MAX_W:
        scale = DETECT_DOWNSCALE_MAX_W / w
        bgr_small = cv2.resize(bgr, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
    else:
        bgr_small = bgr

    gray = cv2.cvtColor(bgr_small, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5)
    if len(faces) == 0:
        return None

    x, y, fw, fh = max(faces, key=lambda r: r[2]*r[3])

    if scale != 1.0:
        inv = 1.0 / scale
        x, y, fw, fh = int(x*inv), int(y*inv), int(fw*inv), int(fh*inv)

    pad = int(0.20 * fw)
    x1 = max(0, x - pad); y1 = max(0, y - pad)
    x2 = min(bgr.shape[1], x + fw + pad)
    y2 = min(bgr.shape[0], y + fh + pad)

    face = bgr[y1:y2, x1:x2]
    if face.size == 0:
        return None
    return face


# ----------------------------
# Load compact DB (prototypes)
# ----------------------------
if not os.path.isfile(DB_COMPACT_PATH):
    raise FileNotFoundError(f"Missing compact DB: {DB_COMPACT_PATH}")

db_npz = np.load(DB_COMPACT_PATH, allow_pickle=True)

proto_mat = []
proto_name = []
for name in db_npz.files:
    E = db_npz[name].astype(np.float32)  # (K,512)
    for row in E:
        proto_mat.append(row)
        proto_name.append(name)

proto_mat = np.stack(proto_mat, axis=0)
print("✅ Loaded compact DB:", DB_COMPACT_PATH)
print("   Identities:", sorted(set(proto_name)))
print("   Total prototypes:", proto_mat.shape[0])

# ----------------------------
# Load model
# ----------------------------
if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Missing model file: {MODEL_PATH}")

itp = tflite.Interpreter(model_path=MODEL_PATH, num_threads=4)
itp.allocate_tensors()
inp = itp.get_input_details()[0]
out = itp.get_output_details()[0]
H, W = int(inp["shape"][1]), int(inp["shape"][2])

print("✅ Using:", USING)
print("Model input:", inp["shape"], inp["dtype"], "output:", out["shape"], out["dtype"])

def preprocess(face_bgr):
    rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (W, H), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    x = (rgb - 127.5) / 128.0
    return x[None, ...]

def embed(face_bgr):
    x = preprocess(face_bgr).astype(np.float32)
    itp.set_tensor(inp["index"], x)
    itp.invoke()
    e = itp.get_tensor(out["index"])[0].astype(np.float32)
    return l2_normalize(e)

# ----------------------------
# Tune threshold on unknown images
# ----------------------------
if not os.path.isdir(UNKNOWN_DIR):
    raise FileNotFoundError(f"Unknown folder not found: {UNKNOWN_DIR}")

unk_imgs = list_images(UNKNOWN_DIR)
if len(unk_imgs) == 0:
    raise RuntimeError(f"Unknown folder is empty: {UNKNOWN_DIR}")

random.shuffle(unk_imgs)
if MAX_UNKNOWN_IMGS is not None:
    unk_imgs = unk_imgs[:MAX_UNKNOWN_IMGS]

print(f"\nTuning threshold using unknown images: {len(unk_imgs)} images")
unk_best_sims = []
unk_skipped = 0

t0 = time.time()

for i, img in enumerate(unk_imgs, start=1):
    if (i % PRINT_EVERY) == 0 or i == len(unk_imgs):
        print_progress("  unknown", i, len(unk_imgs), extra=f"collected={len(unk_best_sims)} skip={unk_skipped}")

    bgr = cv2.imread(os.path.join(UNKNOWN_DIR, img))
    if bgr is None:
        unk_skipped += 1
        continue

    face = detect_biggest_face(bgr)
    if face is None:
        unk_skipped += 1
        continue

    try:
        e = embed(face)
    except Exception:
        unk_skipped += 1
        continue

    sims = proto_mat @ e
    best = float(np.max(sims))
    unk_best_sims.append(best)

unk_best_sims = np.array(unk_best_sims, dtype=np.float32)
if len(unk_best_sims) == 0:
    raise RuntimeError("No usable unknown faces were embedded. Check detection / images.")

threshold = float(np.quantile(unk_best_sims, UNKNOWN_REJECT_QUANTILE))

elapsed = time.time() - t0
print(f"\n✅ Threshold tuned on COMPACT DB")
print(f"   quantile={UNKNOWN_REJECT_QUANTILE}  => threshold={threshold:.3f}")
print(f"   unknown best-sim mean={unk_best_sims.mean():.3f}, max={unk_best_sims.max():.3f}")
print(f"   usable={len(unk_best_sims)}  skipped={unk_skipped}")
print(f"⏱️ Done in {elapsed/60:.1f} minutes")

# Save
with open(THRESH_OUT, "w") as f:
    json.dump({
        "threshold": float(threshold),
        "quantile": UNKNOWN_REJECT_QUANTILE,
        "model_path": MODEL_PATH,
        "db_path": DB_COMPACT_PATH,
        "unknown_dir": UNKNOWN_DIR,
        "input_norm": "(rgb-127.5)/128.0",
        "embedding_dim": int(out["shape"][-1]),
        "unknown_stats": {
            "usable": int(len(unk_best_sims)),
            "skipped": int(unk_skipped),
            "mean_best_sim": float(unk_best_sims.mean()),
            "max_best_sim": float(unk_best_sims.max())
        }
    }, f, indent=2)

print(f"✅ Saved: {THRESH_OUT}")



✅ Loaded compact DB: face_db_compact.npz
   Identities: ['barhoumi_rami', 'zkaria_arfaoui']
   Total prototypes: 2
✅ Using: tensorflow.lite
Model input: [  1 160 160   3] <class 'numpy.float32'> output: [  1 512] <class 'numpy.float32'>

Tuning threshold using unknown images: 8473 images
  unknown: 8473/8473 (100.0%) collected=6831 skip=1641

✅ Threshold tuned on COMPACT DB
   quantile=0.99  => threshold=0.504
   unknown best-sim mean=0.125, max=0.871
   usable=6832  skipped=1641
⏱️ Done in 4.1 minutes
✅ Saved: threshold_compact.json
